# M4 Baseline - Arabic Flood-Crisis Tweet Classification

Single-agent baseline: one Gemini agent makes three sequential calls (initial classification, self-review, final classification).

Before you start: add your Gemini API key to **Colab Secrets** (key icon on the left) with the name `GEMINI_API_KEY`, and turn on notebook access.

## 1. Get the code and install packages

In [ ]:
!git clone https://github.com/Maya-mohammed/arabic-flood-m4-baseline.git
%cd arabic-flood-m4-baseline
!pip install -q -r requirements.txt pytest

## 2. Load the API key from Colab Secrets (it is never printed)

In [ ]:
import os
from google.colab import userdata
key = userdata.get("GEMINI_API_KEY")
if not key:
    raise RuntimeError("GEMINI_API_KEY is missing from Colab Secrets")
os.environ["GEMINI_API_KEY"] = key
print("Gemini key loaded")

## 3. Get the dataset from the original authors and prepare the splits

The corpus is downloaded from the authors' public repository. It is kept in this Colab session only and is not added to our repository.

In [ ]:
!git clone -q --depth 1 https://github.com/alaa-a-a/Arabic-Twitter-corpus-for-flood-detection.git source_data
!mkdir -p private_data && cp source_data/ArabicFloods.csv private_data/
!python prepare_data.py --csv private_data/ArabicFloods.csv

## 4. Offline unit tests (no API calls)

In [ ]:
!python -m pytest -q tests

## 5. Classify one tweet

In [ ]:
from baseline import classify_tweet
r = classify_tweet("السيل قطع الطريق بين القرية والمدينة، انتبهوا ولا تمرون من مجرى الوادي")
print("Initial:    ", r["initial_prediction"])
print("Self-review:", r["self_review_prediction"])
print("Final:      ", r["relevance"], "/", r["information_type"])
print("Calls:", r["successful_model_calls"], "| Repairs:", r["repairs"], "| Time:", r["latency_seconds"], "s | Tokens:", r["total_tokens"])

## 6. Run the labelled test cases (real API calls)

`--pause 15` waits 15 seconds between tweets so the free-tier per-minute limit is not used up. This takes about 5 minutes.

In [ ]:
!python evaluate.py --cases private_data/normal_cases.csv --out private_data/normal_results.csv --pause 15
!python evaluate.py --cases edge_cases_labelled_m4.csv --out private_data/edge_results.csv --pause 15

## 7. User interface

Run the cell, then type a new tweet or click one of the test cases. `share=True` also gives a temporary public link; anyone with the link uses your API key, so stop the cell after the demo.

In [ ]:
import app
app.demo.launch(share=True)